# Agent Memory

## Scenario: retain useful Acme incident knowledge without retaining a bad diagnosis

This notebook treats memory as a governed write → manage → read subsystem. It uses a deterministic store, so the learning path has no model, vector database, or credential dependency.

**Outcomes:** distinguish four memory types, apply scope/provenance/privacy contracts, rank retrieval, consolidate and forget contradictions, and keep only selected memory in working context.

![Agent memory lifecycle](assets/memory-lifecycle.svg)

The model may propose an insight; policy decides whether it becomes memory. Retrieval is constrained by namespace before relevance. Memory never gives a model permission to act.

## 1. Taxonomy and boundaries

Working memory is current thread state. Episodic memory is prior tasks/events. Semantic memory is verified knowledge. Procedural memory is approved strategy/policy. A context window holds only selected tokens for one inference; long-term memory is external data and requires scope, freshness, retention, and retrieval policy.

Vector memory is useful for semantic ranking, structured memory for precise facts and preferences, graphs for relationships, and temporal logs for sequence/replay. In production, hybrid retrieval filters scope and metadata first, then ranks candidates.

In [ ]:
from lab import MemoryType, run_demo, seed_store

store = seed_store()
for memory in store.records.values():
    print(memory.id, memory.type.value, memory.namespace, memory.confidence, memory.text)

## 2. Read: ranking is not authorization

A robust memory read uses tenant/user/project namespace first, then type, retention, freshness, source trust, and finally relevance/importance/confidence. A semantically similar memory from another tenant must not be eligible. Context should contain compact IDs and attributed facts, not every record.

The lab retrieves Acme payment/EU memories and demonstrates that an unverified old diagnosis can rank—making management and contradiction resolution necessary.

In [ ]:
store = seed_store()
before = store.retrieve(('tenant', 'acme'), {'payments', 'eu'})
print([memory.id for memory in before])
assert 'wrong-old' in [memory.id for memory in before]

## 3. Write, consolidate, forget, and resolve contradictions

Write only typed, attributable, scoped items. A reflection can propose an update after a task, but evidence and policy must validate it. Do not silently overwrite contradictions: expire/supersede the old item with a reason, link the new verified item, and keep the audit trail. Forgetting is a feature: retention expiry, user deletion, policy change, stale knowledge, and disproved facts all require removal from future retrieval.

Personalization requires privacy boundaries: separate per-user from tenant-wide facts, minimize sensitive data, support deletion, and key caches by scope/policy/version.

In [ ]:
store = run_demo()
after = store.retrieve(('tenant', 'acme'), {'payments', 'eu'})
print([memory.id for memory in after])
print('\nAudit:')
print('\n'.join(store.audit))
assert 'wrong-old' not in [memory.id for memory in after]
assert 'fact-2' in [memory.id for memory in after]

## Production practices, evaluation, and exercises

Evaluate write precision, retrieval recall/precision, stale-memory rate, contradiction handling, leakage rate, deletion/expiry correctness, personalized outcome lift, token/cost impact, and downstream action quality. Test memory poisoning, cross-tenant retrieval, outdated facts, duplicate writes, lost audit links, and recovery after an interrupted consolidation.

**Exercises:** add `globex` memory and prove it cannot be returned for Acme; add expiry and test forgetting; implement time-decay; add a reflection proposal with no evidence and reject it; compare vector-only, structured-only, graph, temporal, and hybrid retrieval.

## Sources

- [Anthropic context engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)
- [LangGraph memory overview](https://docs.langchain.com/oss/python/concepts/memory)
- [MemGPT](https://arxiv.org/abs/2310.08560)
- [Generative Agents](https://arxiv.org/abs/2304.03442)
- [LLM autonomous-agent survey](https://arxiv.org/abs/2308.11432)